# Recommender System - Content-Based Filtering
This notebook documents a practical content-based filtering pipeline using the TMDB 5000 dataset.


## Objective
Build a feature representation for each movie and recommend similar items based on content similarity.


In [1]:
from src import (
    load_movies, load_credits, merge_datasets,
    build_features, vectorize_features,
    compute_item_similarity, recommend_content_based
)


## Data Loading
TMDB 5000 provides movie metadata and credits (cast/crew). We merge both sources for feature extraction.


In [2]:
movies_df = load_movies("datasets/tmdb_5000_movies.csv")
credits_df = load_credits("datasets/tmdb_5000_credits.csv")
tmdb_df = merge_datasets(movies_df, credits_df)


In [3]:
from IPython.display import display
display(tmdb_df.head())


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count,cast,crew,title
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,7.2,11800,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",Avatar
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",6.9,4500,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",Pirates of the Caribbean: At World's End
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,6.3,4466,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",Spectre
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,7.6,9106,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",The Dark Knight Rises
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",...,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",6.1,2124,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",John Carter


## Data Exploration
Quick checks for dataset size and missing values before feature engineering.


In [4]:
tmdb_df.shape
tmdb_df[["title", "overview", "genres", "keywords", "cast", "crew"]].isna().sum()


title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

## Feature Engineering
Extract genres, keywords, cast, director, and overview, then combine them into a single text feature.


In [5]:
tmdb_features = build_features(tmdb_df)
tmdb_features[["title", "features"]].head()


,title,features
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca..."


## Vectorization
Convert the combined text features into numerical vectors using TF-IDF.


In [6]:
feature_matrix, vectorizer = vectorize_features(tmdb_features)


## Similarity Computation
Compute cosine similarity between all movie feature vectors.


In [7]:
similarity_matrix = compute_item_similarity(feature_matrix)


## Recommendation Logic
Select a target movie title and retrieve the most similar items.


In [8]:
recommendations = recommend_content_based(
    title="Avatar",
    df=tmdb_features,
    similarity_matrix=similarity_matrix,
    top_n=10
)


## Example
This section shows the top recommended movies for a single title using content similarity.


In [9]:
display(recommendations)


,id,title,score
0,679,Aliens,0.309345
1,8077,Alien³,0.275817
2,2067,Mission to Mars,0.253985
3,698,Moonraker,0.253971
4,348,Alien,0.246747
5,9016,Treasure Planet,0.226065
6,957,Spaceballs,0.224540
7,11954,Lifeforce,0.218406
8,869,Planet of the Apes,0.214592
9,81796,Lockout,0.212143


## Limitation
Content-based filtering can over-specialize and may miss diverse or novel recommendations.
It also depends heavily on the quality of the available metadata.
